In [ ]:
import asyncio
from typing import List, Dict, Any
from pydantic import BaseModel, Field
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from taxonomy import HARDCODED_SKILL_TAXONOMY, random

search_tool = DuckDuckGoSearchRun()

class MCQQuestion(BaseModel):
    subtopic: str = Field(description="The specific subtopic being tested.")
    question_text: str = Field(description="The technical multiple-choice question.")
    option_a: str
    option_b: str
    option_c: str
    option_d: str
    correct_answer: str = Field(description="Must be exactly 'A', 'B', 'C', or 'D'.")
    explanation: str = Field(description="Brief explanation of why the answer is correct for the analysis report.")


In [ ]:
async def generate_single_question(skill: str, subtopic: str,structured_llm) -> MCQQuestion:
    """
    Fetches real-world context for a subtopic and generates a structured MCQ.
    """
    try:
        # Step 1: Web Search (RAG Context Fetching)
        search_query = f"{skill} {subtopic} best practices documentation"
        context = search_tool.invoke(search_query)
        
        # Step 2: LLM Generation
        prompt_template = PromptTemplate.from_template("""
        You are an expert technical interviewer assessing a senior developer.
        Generate a highly technical multiple-choice question for the skill: {skill}.
        Specific subtopic: {subtopic}.
        
        Use this recent web context to make the question realistic and modern:
        {context}
        
        The question should test deep understanding, not just basic definitions.
        """)
        
        return await structured_llm.ainvoke(prompt_template.format(
            skill=skill, 
            subtopic=subtopic, 
            context=context[:1500]
        ))
    except Exception as e:
        print(f"Failed generating question for {subtopic}: {e}")
        # Fallback question if network/API fails
        return MCQQuestion(
            subtopic=subtopic,
            question_text=f"What is the primary function of {subtopic} in {skill}?",
            option_a="Configuration", option_b="Optimization", option_c="Execution", option_d="Logging",
            correct_answer="B", explanation="General fallback execution."
        )

In [ ]:
async def generate_skill_test(skill: str,llm) -> List[MCQQuestion]:
    """
    Randomly selects 10 subtopics and generates the test asynchronously.
    """
    if skill not in HARDCODED_SKILL_TAXONOMY:
        raise ValueError("Skill taxonomy not found.")
    
    structured_llm = llm.with_structured_output(MCQQuestion)
    # Select 10 random unique subtopics
    selected_subtopics = random.sample(HARDCODED_SKILL_TAXONOMY[skill], 10)
    
    # Run all 10 RAG generations simultaneously
    tasks = [generate_single_question(skill, subtopic,structured_llm) for subtopic in selected_subtopics]
    questions = await asyncio.gather(*tasks)
    
    return questions

In [ ]:

def evaluate_test(questions: List[Dict[str, Any]], user_answers: Dict[str, str]) -> Dict[str, Any]:
    """
    Compares user answers to correct answers and generates the final badge/report.
    user_answers format: {"0": "A", "1": "C", ...}
    """
    score = 0
    total = len(questions)
    analysis_report = []
    
    for index, q in enumerate(questions):
        user_choice = user_answers.get(str(index), "").upper()
        correct_choice = q["correct_answer"].upper()
        
        is_correct = (user_choice == correct_choice)
        if is_correct:
            score += 1
            
        analysis_report.append({
            "subtopic": q["subtopic"],
            "question": q["question_text"],
            "user_answer": user_choice,
            "correct_answer": correct_choice,
            "is_correct": is_correct,
            "explanation": q["explanation"]
        })
        
    percentage = (score / total) * 100
    passed = percentage >= 70.0 # 70% threshold to pass
    
    return {
        "status": "Evaluated",
        "score": percentage,
        "passed": passed,
        "awarded_badge": "Verified" if passed else None,
        "action_required": "None" if passed else "Retry Option / Study Report",
        "analysis_report": analysis_report
    }